新しい形状の模索2

案4：モーメント不変量 (Hu Moments)
Hu Momentsは、物体の回転、拡大縮小、平行移動に対して変わらない（不変な）7つの数値で形状を記述します。これらの数値のどれかが「AB対C」の分類に強く効く可能性があります。

In [9]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# --- 1. Hu Moments (案4) の分析クラス ---

class KeijoAnalyzer_HuMoments_Flusser:
    """
    シイタケのマスク画像から形状特徴（Hu Moments + Flusserの8番目）を分析する
    """
    def __init__(self):
        pass

    def analyze_hu_moments(self, img_path):
        """
        提案4：7つのHu MomentsとFlusserの8番目の不変量（h7）を計算し、Log変換して返す
        """
        mask = cv2.imread(img_path)
        if mask is None:
            print(f"⚠️ 読み込み失敗: {img_path}")
            return None

        gray = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
        _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            print(f"⚠️ 輪郭なし: {img_path}")
            return None

        max_contour = max(contours, key=cv2.contourArea)
        
        if cv2.contourArea(max_contour) < 10:
             print(f"⚠️ 面積が小さすぎる: {img_path}")
             return None

        # 輪郭からモーメントを計算
        M = cv2.moments(max_contour)
        
        if M["m00"] == 0:
            return None
            
        # Hu Moments (phi_1〜phi_7) を計算
        hu = cv2.HuMoments(M)
        
        # 1. 標準の7つの不変量 (h0〜h6, Log変換)
        # sgn(hu) * log10(|hu|) で、サインを保持しつつスケールを調整
        log_hu_standard = -np.sign(hu) * np.log10(np.abs(hu))
        
        # 2. Flusserの8番目の不変量 (h7) を追加
        # $\phi_7$ (hu[6]) の絶対値を取り、Log変換することで反射不変とする
        phi7_abs = np.abs(hu[6])
        # $\phi_7$は通常非常に小さいため、Log変換でスケールを調整
        log_hu_8th = -np.log10(phi7_abs)
        
        # 3. 8つの不変量を結合して返す (h0〜h7)
        final_hu_vector = np.append(log_hu_standard, log_hu_8th)

        return final_hu_vector.flatten()


class Keijo_folder_HuMoments(KeijoAnalyzer_HuMoments_Flusser):
    """
    指定されたフォルダ内の全画像のHu Moments (h0〜h7)を計算し、CSVに保存する
    """
    def __init__(self, base_dir, mask_dir="mask", subfolder="collage_1", output_csv=None):
        super().__init__()
        self.folder_path = os.path.join(base_dir, mask_dir, subfolder)
        self.output_csv = output_csv
        self.results_list = []

    def run(self):
        image_paths = glob.glob(os.path.join(self.folder_path, "*.jpg")) + \
                      glob.glob(os.path.join(self.folder_path, "*.png"))
        
        print(f"📂 フォルダ(Hu Moments + h7): {self.folder_path} に画像 {len(image_paths)} 枚")
        
        self.results_list = []

        for img_path in image_paths:
            file_name = os.path.basename(img_path)
            
            # Hu Moments (h0〜h7) を計算
            hu_vector = self.analyze_hu_moments(img_path)
            
            if hu_vector is not None:
                # ファイル名と8つのモーメントを結合してリストに追加
                row = [file_name] + list(hu_vector)
                self.results_list.append(row)
            else:
                self.results_list.append([file_name] + [np.nan] * 8) # 8つのNaNを追加
                
        return self.results_list

    def save_results(self):
        # h0 から h7 までの8列
        column_names = ['filename'] + [f'h{i}' for i in range(8)]
        df = pd.DataFrame(self.results_list, columns=column_names)
        df = df.set_index('filename')
        
        df.to_csv(self.output_csv)

        if df.empty:
            print(f"⚠️ 該当フォルダに画像がなかったため、空のCSVを保存しました: {self.output_csv}")
        else:
            print(f"✅ 結果を保存: {self.output_csv}")


# --- 2. t検定と (A+B) vs C グラフ作成の関数 (Hu Moments版) ---

def run_t_test_hu_moments(csv_base_dir, categories_ab, category_c):
    """
    Hu Moments (h0〜h7) のCSVを読み込み、最も分類に適した特徴量を特定し、t検定を実行する
    """
    print("\n--- t検定 (Hu Moments + h7) 実行 ---")
    
    all_data_list = []
    
    # 1. 全データを読み込み・結合
    for category in categories_ab + [category_c]:
        csv_path = os.path.join(csv_base_dir, f"keijo_hu_moments_{category}.csv")
        try:
            df = pd.read_csv(csv_path)
            df['category'] = 'AB' if category in categories_ab else 'C'
            all_data_list.append(df)
        except FileNotFoundError:
            print(f"❌ CSVファイルが見つかりません: {csv_path}")
            return
    
    combined_df = pd.concat(all_data_list, ignore_index=True)
    combined_df = combined_df.dropna() # NaNを含む行は除外

    if combined_df.empty:
        print("❌ 有効なデータが空のため、t検定を実行できません。")
        return

    # 2. 8つの特徴量すべてについて t検定を実行 (h0 から h7 まで)
    best_t_abs = 0
    best_feature = None
    best_p_value = 1.0

    # ループ範囲を 8つ (h0〜h7) に変更
    for i in range(8):
        feature = f'h{i}'
        
        # CSVにh7の列がない場合（旧版のCSVを使用した場合など）のチェック
        if feature not in combined_df.columns:
            print(f"⚠️ 特徴量 {feature} がデータフレームに見つかりません。")
            continue

        group_ab = combined_df[combined_df['category'] == 'AB'][feature].values
        group_c = combined_df[combined_df['category'] == 'C'][feature].values

        if len(group_ab) < 2 or len(group_c) < 2: continue

        t_statistic, p_value = stats.ttest_ind(group_ab, group_c, equal_var=False)
        
        if abs(t_statistic) > best_t_abs:
            best_t_abs = abs(t_statistic)
            best_t_statistic = t_statistic
            best_p_value = p_value
            best_feature = feature
            best_group_ab = group_ab
            best_group_c = group_c

    print("\n--- t検定結果 (Hu Moments + Flusser h7) ---")
    if best_feature:
        print(f"✅ 最適な分類特徴量: {best_feature} (h7は反射不変)")
        print(f"t値 (statistic): {best_t_statistic:.4f}")
        print(f"p値 (p-value): {best_p_value}")
        print(f"t値の絶対値（{best_t_abs:.4f}）が最も大きいです。")
        if best_p_value < 0.05:
            print("p値が0.05より小さいため、この特徴量には「統計的に有意な差がある」と判断できます。")
        else:
            print("統計的な差があるとは言い切れません。")

        # 3. 最適特徴量の箱ひげ図を作成・保存
        output_graph_path = os.path.join(csv_base_dir, f"hu_moments_{best_feature}_AB_vs_C_boxplot.png")

        plt.figure(figsize=(4, 6))
        bplot = plt.boxplot([best_group_ab, best_group_c], labels=["A+B", "C"], patch_artist=True)
        colors = ['lightblue', 'lightgreen']
        for patch, color in zip(bplot['boxes'], colors):
            patch.set_facecolor(color)
        plt.title(f'Hu Moment {best_feature} Value: (A+B) vs C')
        plt.ylabel(f'Log Hu Moment Value')
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.savefig(output_graph_path)
        print(f"\n✅ 最適特徴量の比較グラフを保存しました: {output_graph_path}")
        plt.close()

    else:
        print("❌ t検定を実行できる有効なデータが見つかりませんでした。")


# --- 3. メイン実行ブロック ---

if __name__ == "__main__":
    
    # --- 設定: 入力元と出力先 ---
    INPUT_DATA_DIR = "/home/data/1124_keijotest"
    OUTPUT_DIR = "/home/src/keijo/1116_newway"
    CATEGORIES = ["A", "B", "C"]
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # --- STEP 1: 分析の実行 (Hu Moments + h7) ---
    print("--- 分析実行 (Hu Moments + h7) ---")
    
    for category in CATEGORIES:
        output_csv_path = os.path.join(OUTPUT_DIR, f"keijo_hu_moments_{category}.csv")
        
        analyzer = Keijo_folder_HuMoments(
            base_dir=INPUT_DATA_DIR,
            mask_dir="mask",
            subfolder=category,
            output_csv=output_csv_path
        )
        analyzer.run()
        analyzer.save_results()

    # --- STEP 2: t検定とグラフ作成 (AB vs C) の実行 ---
    run_t_test_hu_moments(
        csv_base_dir=OUTPUT_DIR,
        categories_ab=["A", "B"],
        category_c="C"
    )

--- 分析実行 (Hu Moments + h7) ---
📂 フォルダ(Hu Moments + h7): /home/data/1124_keijotest/mask/A に画像 496 枚
✅ 結果を保存: /home/src/keijo/1116_newway/keijo_hu_moments_A.csv
📂 フォルダ(Hu Moments + h7): /home/data/1124_keijotest/mask/B に画像 302 枚
✅ 結果を保存: /home/src/keijo/1116_newway/keijo_hu_moments_B.csv
📂 フォルダ(Hu Moments + h7): /home/data/1124_keijotest/mask/C に画像 195 枚
✅ 結果を保存: /home/src/keijo/1116_newway/keijo_hu_moments_C.csv

--- t検定 (Hu Moments + h7) 実行 ---

--- t検定結果 (Hu Moments + Flusser h7) ---
✅ 最適な分類特徴量: h7 (h7は反射不変)
t値 (statistic): 10.9785
p値 (p-value): 1.407044280937768e-24
t値の絶対値（10.9785）が最も大きいです。
p値が0.05より小さいため、この特徴量には「統計的に有意な差がある」と判断できます。

✅ 最適特徴量の比較グラフを保存しました: /home/src/keijo/1116_newway/hu_moments_h7_AB_vs_C_boxplot.png


/tmp/ipykernel_1124923/1999447663.py:185: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bplot = plt.boxplot([best_group_ab, best_group_c], labels=["A+B", "C"], patch_artist=True)


案5

In [10]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats 
from sklearn.metrics import mean_squared_error # 今回は不要

# --- 1. フーリエ記述子 (案5) の分析クラス ---

class KeijoAnalyzer_Fourier:
    """
    シイタケのマスク画像から形状特徴（フーリエ記述子）を分析する
    """
    def __init__(self):
        pass

    def analyze_fourier_descriptors(self, img_path, num_descriptors=4):
        """
        提案5：フーリエ記述子 (2次〜5次) を計算し、正規化して返す
        """
        mask = cv2.imread(img_path)
        if mask is None:
            return None

        gray = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)
        _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        contours, _ = cv2.findContours(th, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            return None

        max_contour = max(contours, key=cv2.contourArea)
        
        if len(max_contour) < 10: # 点が少なすぎるとFFTの信頼性が低い
             return None

        # 輪郭座標を複素数配列に変換 (x + j*y)
        contour_array = max_contour.squeeze().astype(np.float64)
        complex_contour = contour_array[:, 0] + 1j * contour_array[:, 1]

        # FFT (高速フーリエ変換) を実行
        fourier_coeffs = np.fft.fft(complex_contour)

        # 記述子の絶対値を取得
        F_abs = np.abs(fourier_coeffs)
        
        # F1 (|F_1|) をスケール（回転・拡大縮小不変性）の基準とする
        F1_abs = F_abs[1]

        if F1_abs == 0: 
            return None
        
        # 2次から (num_descriptors + 1) 次までを抽出し、F1で正規化
        # [F_2/F_1, F_3/F_1, F_4/F_1, F_5/F_1] を計算
        descriptors = F_abs[2 : 2 + num_descriptors] / F1_abs
        
        return descriptors.flatten() # 4次元のNumPy配列を返す


class Keijo_folder_Fourier(KeijoAnalyzer_Fourier):
    """
    指定されたフォルダ内の全画像のフーリエ記述子を計算し、CSVに保存する
    """
    def __init__(self, base_dir, mask_dir="mask", subfolder="collage_1", output_csv=None):
        super().__init__()
        self.folder_path = os.path.join(base_dir, mask_dir, subfolder)
        self.output_csv = output_csv
        self.results_list = []

    def run(self):
        image_paths = glob.glob(os.path.join(self.folder_path, "*.jpg")) + \
                      glob.glob(os.path.join(self.folder_path, "*.png"))
        
        print(f"📂 フォルダ(フーリエ): {self.folder_path} に画像 {len(image_paths)} 枚")
        
        self.results_list = []

        for img_path in image_paths:
            file_name = os.path.basename(img_path)
            
            fourier_vector = self.analyze_fourier_descriptors(img_path)
            
            if fourier_vector is not None:
                # ファイル名と4つの記述子を結合してリストに追加
                row = [file_name] + list(fourier_vector)
                self.results_list.append(row)
            else:
                self.results_list.append([file_name] + [np.nan] * 4) # 4つのNaNを追加
                
        return self.results_list

    def save_results(self):
        # 列名: f2_norm, f3_norm, f4_norm, f5_norm
        column_names = ['filename'] + [f'f{i}_norm' for i in range(2, 6)]
        df = pd.DataFrame(self.results_list, columns=column_names)
        df = df.set_index('filename')
        
        df.to_csv(self.output_csv)

        if df.empty:
            print(f"⚠️ 該当フォルダに画像がなかったため、空のCSVを保存しました: {self.output_csv}")
        else:
            print(f"✅ 結果を保存: {self.output_csv}")


# --- 4. t検定と (A+B) vs C グラフ作成の関数 (フーリエ記述子版) ---

def run_t_test_fourier(csv_base_dir, categories_ab, category_c):
    """
    フーリエ記述子のCSVを読み込み、最も分類に適した特徴量を特定し、t検定を実行する
    """
    print("\n--- t検定 (フーリエ記述子) 実行 ---")
    
    # 1. 全データを読み込み・結合
    all_data_list = []
    for category in categories_ab + [category_c]:
        csv_path = os.path.join(csv_base_dir, f"keijo_fourier_{category}.csv")
        try:
            df = pd.read_csv(csv_path)
            df['category'] = 'AB' if category in categories_ab else 'C'
            all_data_list.append(df)
        except FileNotFoundError:
            print(f"❌ CSVファイルが見つかりません: {csv_path}")
            return
    
    combined_df = pd.concat(all_data_list, ignore_index=True)
    combined_df = combined_df.dropna() # NaNを含む行は除外

    if combined_df.empty:
        print("❌ 有効なデータが空のため、t検定を実行できません。")
        return

    # 2. 4つの特徴量すべてについて t検定を実行
    best_t_abs = 0
    best_feature = None
    best_p_value = 1.0

    # f2_norm, f3_norm, f4_norm, f5_norm の4つを評価
    for i in range(2, 6):
        feature = f'f{i}_norm'
        group_ab = combined_df[combined_df['category'] == 'AB'][feature].values
        group_c = combined_df[combined_df['category'] == 'C'][feature].values

        if len(group_ab) < 2 or len(group_c) < 2: continue

        t_statistic, p_value = stats.ttest_ind(group_ab, group_c, equal_var=False)
        
        if abs(t_statistic) > best_t_abs:
            best_t_abs = abs(t_statistic)
            best_t_statistic = t_statistic
            best_p_value = p_value
            best_feature = feature
            best_group_ab = group_ab
            best_group_c = group_c

    print("\n--- t検定結果 (フーリエ記述子) ---")
    if best_feature:
        print(f"✅ 最適な分類特徴量: {best_feature}")
        print(f"t値 (statistic): {best_t_statistic:.4f}")
        print(f"p値 (p-value): {best_p_value}")
        print(f"t値の絶対値（{best_t_abs:.4f}）が最も大きいです。")
        if best_p_value < 0.05:
            print("p値が0.05より小さいため、この特徴量には「統計的に有意な差がある」と判断できます。")
        else:
            print("統計的な差があるとは言い切れません。")

        # 3. 最適特徴量の箱ひげ図を作成・保存
        output_graph_path = os.path.join(csv_base_dir, f"fourier_{best_feature}_AB_vs_C_boxplot.png")

        plt.figure(figsize=(4, 6))
        bplot = plt.boxplot([best_group_ab, best_group_c], labels=["A+B", "C"], patch_artist=True)
        colors = ['lightblue', 'lightgreen']
        for patch, color in zip(bplot['boxes'], colors):
            patch.set_facecolor(color)
        plt.title(f'Fourier Descriptor {best_feature}: (A+B) vs C')
        plt.ylabel(f'Normalized Fourier Magnitude')
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.savefig(output_graph_path)
        print(f"\n✅ 最適特徴量の比較グラフを保存しました: {output_graph_path}")
        plt.close()

    else:
        print("❌ t検定を実行できる有効なデータが見つかりませんでした。")


# --- 4. メイン実行ブロック ---

if __name__ == "__main__":
    
    # --- 設定: 入力元と出力先を分離 ---
    INPUT_DATA_DIR = "/home/data/1124_keijotest"
    OUTPUT_DIR = "/home/src/keijo/1116_newway"
    CATEGORIES = ["A", "B", "C"]
    
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    # --- STEP 1: 分析の実行 (フーリエ記述子) ---
    print("--- 分析実行 (フーリエ記述子) ---")
    
    for category in CATEGORIES:
        output_csv_path = os.path.join(OUTPUT_DIR, f"keijo_fourier_{category}.csv")
        
        analyzer = Keijo_folder_Fourier(
            base_dir=INPUT_DATA_DIR,
            mask_dir="mask",
            subfolder=category,
            output_csv=output_csv_path
        )
        analyzer.run()
        analyzer.save_results()

    # --- STEP 2: t検定とグラフ作成 (AB vs C) の実行 ---
    run_t_test_fourier(
        csv_base_dir=OUTPUT_DIR,
        categories_ab=["A", "B"],
        category_c="C"
    )

--- 分析実行 (フーリエ記述子) ---
📂 フォルダ(フーリエ): /home/data/1124_keijotest/mask/A に画像 496 枚
✅ 結果を保存: /home/src/keijo/1116_newway/keijo_fourier_A.csv
📂 フォルダ(フーリエ): /home/data/1124_keijotest/mask/B に画像 302 枚
✅ 結果を保存: /home/src/keijo/1116_newway/keijo_fourier_B.csv
📂 フォルダ(フーリエ): /home/data/1124_keijotest/mask/C に画像 195 枚
✅ 結果を保存: /home/src/keijo/1116_newway/keijo_fourier_C.csv

--- t検定 (フーリエ記述子) 実行 ---

--- t検定結果 (フーリエ記述子) ---
✅ 最適な分類特徴量: f3_norm
t値 (statistic): 5.6213
p値 (p-value): 3.044923479948659e-08
t値の絶対値（5.6213）が最も大きいです。
p値が0.05より小さいため、この特徴量には「統計的に有意な差がある」と判断できます。

✅ 最適特徴量の比較グラフを保存しました: /home/src/keijo/1116_newway/fourier_f3_norm_AB_vs_C_boxplot.png


/tmp/ipykernel_1124923/3646592199.py:173: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bplot = plt.boxplot([best_group_ab, best_group_c], labels=["A+B", "C"], patch_artist=True)


案6:LDA

In [5]:
import os
import pandas as pd
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.model_selection import cross_val_score, KFold # 分類精度の評価
from sklearn.preprocessing import StandardScaler # データの前処理

def run_lda_hu_moments(csv_base_dir, categories_ab, category_c):
    """
    Hu MomentsのCSVを読み込み、LDAにより (A+B) 対 (C) の分類精度を評価する
    """
    print("\n--- Hu Moments: LDAによる分類精度評価 ---")
    
    all_data_list = []
    
    # 1. 全データを読み込み・結合
    for category in categories_ab + [category_c]:
        csv_path = os.path.join(csv_base_dir, f"keijo_hu_moments_{category}.csv")
        try:
            df = pd.read_csv(csv_path)
            # ターゲット変数 (y): AB=0, C=1
            df['target'] = 0 if category in categories_ab else 1
            all_data_list.append(df)
        except FileNotFoundError:
            print(f"❌ CSVファイルが見つかりません: {csv_path}")
            return
    
    combined_df = pd.concat(all_data_list, ignore_index=True)
    combined_df = combined_df.dropna() # NaNを含む行は除外

    if combined_df.empty:
        print("❌ 有効なデータが空のため、LDAを実行できません。")
        return

    # 2. データとターゲットの準備
    # 特徴量 (X): h0からh6
    feature_cols = [f'h{i}' for i in range(7)]
    X = combined_df[feature_cols].values
    y = combined_df['target'].values
    
    print(f"総データ数: {len(X)}")
    print(f"特徴量数: {X.shape[1]}")

    # 3. 前処理 (スケーリング: LDAはスケーリングが必須ではないが、安定性のために推奨)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # 4. LDAの実行とクロスバリデーションによる精度評価
    lda = LDA()
    
    # 5分割交差検証 (K=5) で分類精度を評価
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(lda, X_scaled, y, cv=kfold, scoring='accuracy')

    # 5. 結果の表示
    mean_accuracy = scores.mean()
    
    print("\n--- LDA分類結果 ---")
    print(f"評価手法: 5分割交差検証")
    print(f"特徴量: Hu Moments (h0〜h6)")
    print(f"平均分類精度 (Accuracy): {mean_accuracy:.4f}")
    print(f"標準偏差: {scores.std():.4f}")
    
    print("\n--- 解釈 ---")
    print(f"分類精度が 1.0 に近いほど、Hu Momentsを使ってABとCの平均形状を明確に分離できています。")
    print(f"この精度を、他の特徴量（円形度、偏平率など）のLDA結果と比較することで、最も優れた手法を判断できます。")
    
    return mean_accuracy


# --- 6. メイン実行ブロック ---

if __name__ == "__main__":
    
    # --- 設定 ---
    INPUT_DATA_DIR = "/home/data/1124_keijotest"
    OUTPUT_DIR = "/home/src/keijo/1116_newway"
    
    # Hu Momentsの分析・CSV保存は別途実行済みであると仮定
    
    # --- LDAの実行 ---
    # CSVが存在しない場合は、Hu Momentsの分析プログラムを先に実行してください。
    accuracy = run_lda_hu_moments(
        csv_base_dir=OUTPUT_DIR,
        categories_ab=["A", "B"],
        category_c="C"
    )

    # この結果を、円形度や偏平率のLDA結果と比較します。


--- Hu Moments: LDAによる分類精度評価 ---
総データ数: 993
特徴量数: 7

--- LDA分類結果 ---
評価手法: 5分割交差検証
特徴量: Hu Moments (h0〜h6)
平均分類精度 (Accuracy): 0.8026
標準偏差: 0.0284

--- 解釈 ---
分類精度が 1.0 に近いほど、Hu Momentsを使ってABとCの平均形状を明確に分離できています。
この精度を、他の特徴量（円形度、偏平率など）のLDA結果と比較することで、最も優れた手法を判断できます。
